# Joint ResNet18 + DINOv2 Feature Fusion

## 1. Experiment Overview

### Research Objective
Investigate whether combining CNN and Vision Transformer representations
through end-to-end joint training improves distracted-driver classification.

### Architecture
- CNN backbone: Fine-tuned ResNet18
- Transformer backbone: Fine-tuned DINOv2 ViT-S/14
- Feature dimensions: 512 + 384 = 896
- Fusion: Feature concatenation followed by a classification head
- Output: 10 driving activity classes

### Training Strategy
1. Initialize both backbones from their existing fine-tuned checkpoints.
2. Train a new fusion classifier with both backbones frozen.
3. Unfreeze both backbones for end-to-end joint fine-tuning.
4. Evaluate on the same four unseen validation drivers.

### Evaluation
Accuracy, macro-F1, per-class recall, and error analysis.

**The final test set is excluded from model development.**

In [2]:
import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet_path = Path(
    "../models/deep_learning/driver_based/resnet18/"
    "resolution_384x288/best_model.pt"
)

dino_path = Path(
    "../results/deep_learning/driver_based/dinov2/"
    "independent_comparison/full/best_model.pt"
)

for name, path in [
    ("ResNet18", resnet_path),
    ("DINOv2", dino_path)
]:
    print(f"\n{'=' * 40}")
    print(name)
    print("Exists:", path.exists())
    print("Path:", path)

    if path.exists():
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("Checkpoint type:", type(ckpt).__name__)

        if isinstance(ckpt, dict):
            print("Top-level keys:", list(ckpt.keys())[:15])

            for key in [
                "model_state_dict",
                "state_dict",
                "backbone_state_dict"
            ]:
                if key in ckpt:
                    print(
                        f"{key} sample:",
                        list(ckpt[key].keys())[:5]
                    )


ResNet18
Exists: True
Path: ../models/deep_learning/driver_based/resnet18/resolution_384x288/best_model.pt
Checkpoint type: OrderedDict
Top-level keys: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias']

DINOv2
Exists: True
Path: ../results/deep_learning/driver_based/dinov2/independent_comparison/full/best_model.pt
Checkpoint type: dict
Top-level keys: ['backbone_state_dict', 'classifier_state_dict', 'best_epoch', 'best_macro_f1', 'best_accuracy', 'seed', 'trainable_blocks', 'initialization']
backbone_state_dict sample: ['cls_token', 'pos_embed', 'mask_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias']


## 3. Joint ResNet18 + DINOv2 Architecture

### Objective
Combine representations learned by ResNet18 and DINOv2 into one neural network.

### Architecture
- ResNet18: 512-dimensional feature representation
- DINOv2 ViT-S/14: 384-dimensional feature representation
- Feature fusion: Concatenation (896 dimensions)
- Classification head: Linear(896, 10)

### Initialization
Both backbones are initialized from their previously fine-tuned checkpoints.

### Training Strategy
1. Verify the combined architecture.
2. Train the fusion classifier.
3. Jointly fine-tune both backbones using one shared loss.

**Current step:** Model construction and forward-pass verification only.

In [3]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

# -------------------------------
# Load fine-tuned ResNet18
# -------------------------------
resnet = resnet18(weights=None, num_classes=10)

resnet_weights = torch.load(
    resnet_path,
    map_location="cpu",
    weights_only=True
)

resnet.load_state_dict(resnet_weights, strict=True)

# Remove classification layer to extract 512 features
resnet.fc = nn.Identity()

# -------------------------------
# Load fine-tuned DINOv2
# -------------------------------
dino = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14",
    pretrained=False
)

dino_checkpoint = torch.load(
    dino_path,
    map_location="cpu",
    weights_only=True
)

dino.load_state_dict(
    dino_checkpoint["backbone_state_dict"],
    strict=True
)

print("ResNet18 loaded successfully!")
print("DINOv2 loaded successfully!")

ResNet18 loaded successfully!
DINOv2 loaded successfully!


Using cache found in /home/js/.cache/torch/hub/facebookresearch_dinov2_main
/home/js/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/js/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/js/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [4]:
class ResNetDINOv2Fusion(nn.Module):

    def __init__(self, resnet, dino):
        super().__init__()

        self.resnet = resnet
        self.dino = dino

        # Combine 512 + 384 features
        self.classifier = nn.Linear(896, 10)

    def forward(self, x_resnet, x_dino):

        # CNN features
        resnet_features = self.resnet(x_resnet)

        # Transformer features
        dino_features = self.dino(x_dino)

        # Concatenate features
        combined_features = torch.cat(
            [resnet_features, dino_features],
            dim=1
        )

        # Final classification
        logits = self.classifier(combined_features)

        return logits


# Initialize joint model
joint_model = ResNetDINOv2Fusion(
    resnet,
    dino
).to(device)

print("Joint model initialized successfully!")

Joint model initialized successfully!


## 4. Joint Architecture — Forward-Pass Verification

### Objective
Confirm that ResNet18 and DINOv2 produce compatible feature representations.

### Expected Dimensions
- ResNet18: [2, 512]
- DINOv2: [2, 384]
- Combined representation: [2, 896]
- Final predictions: [2, 10]

No backpropagation or parameter updates occur during this step.

In [5]:
joint_model.eval()

# Simulate two RGB images at each model's resolution
x_resnet = torch.randn(2, 3, 288, 384).to(device)
x_dino = torch.randn(2, 3, 252, 336).to(device)

with torch.inference_mode():

    resnet_features = joint_model.resnet(x_resnet)
    dino_features = joint_model.dino(x_dino)

    predictions = joint_model(x_resnet, x_dino)

print("ResNet18 features:", resnet_features.shape)
print("DINOv2 features:", dino_features.shape)

print(
    "Combined feature dimension:",
    resnet_features.shape[1] + dino_features.shape[1]
)

print("Final predictions:", predictions.shape)

ResNet18 features: torch.Size([2, 512])
DINOv2 features: torch.Size([2, 384])
Combined feature dimension: 896
Final predictions: torch.Size([2, 10])


## 5. Dual-Input Dataset and Augmentation

### Objective
Prepare synchronized inputs for ResNet18 and DINOv2.

### Architecture Inputs
- ResNet18: 384 × 288 pixels
- DINOv2: 336 × 252 pixels

### Augmentation Strategy
Apply the same random geometric and photometric transformations
to both model inputs before resizing.

Augmentation is applied to training images only.

### Generalization
Use stronger but label-preserving augmentations to improve
robustness to changes in lighting, camera position, and appearance.

Horizontal flipping and aggressive cropping are disabled.

### Evaluation
- 18 training drivers
- 4 validation drivers
- Zero driver overlap

In [6]:
import pandas as pd
import torch
from PIL import Image
from pathlib import Path

from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader

# Load our existing splits
split_dir = Path("../data/splits/driver_based")

train_df = pd.read_csv(split_dir / "train.csv")
val_df = pd.read_csv(split_dir / "val.csv")

# Verify driver independence
assert set(train_df["subject"]).isdisjoint(
    set(val_df["subject"])
)

# Image directory
image_root = Path("../data/raw/imgs/train")

assert image_root.exists(), "Image directory not found"

# Shared training augmentation
train_augmentation = transforms.Compose([

    transforms.RandomAffine(
        degrees=7,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
        fill=(30, 30, 30)
    ),

    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.02
    ),

    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 1.0)
        )
    ], p=0.10)
])

# Model-specific preprocessing
def create_preprocessing(height, width):

    return transforms.Compose([
        transforms.Resize((height, width)),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


resnet_transform = create_preprocessing(288, 384)

dino_transform = create_preprocessing(252, 336)

In [7]:
class JointDriverDataset(Dataset):

    def __init__(self, df, image_root, augment=False):

        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = (
            self.image_root /
            row["classname"] /
            row["img"]
        )

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        # Apply shared augmentation once
        if self.augment:
            image = train_augmentation(image)

        # Same image, different resolutions
        resnet_image = resnet_transform(image)
        dino_image = dino_transform(image)

        label = int(row["classname"][1:])

        return resnet_image, dino_image, label


# Training dataset
train_dataset = JointDriverDataset(
    train_df,
    image_root,
    augment=True
)

# Validation dataset
val_dataset = JointDriverDataset(
    val_df,
    image_root,
    augment=False
)

# Training loader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=torch.Generator().manual_seed(42)
)

# Validation loader
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Verify one batch
x_resnet, x_dino, labels = next(iter(train_loader))

print("ResNet18 input:", x_resnet.shape)
print("DINOv2 input:", x_dino.shape)
print("Labels:", labels.shape)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

ResNet18 input: torch.Size([32, 3, 288, 384])
DINOv2 input: torch.Size([32, 3, 252, 336])
Labels: torch.Size([32])
Training batches: 492
Validation batches: 113


In [8]:
import torch
import torch.nn as nn

# Enable gradients for both backbones and fusion classifier
joint_model.requires_grad_(True)

# Evaluation mode prevents BatchNorm running-stat updates.
# Gradients still work because we are NOT using inference_mode.
joint_model.eval()

# Load one actual training batch
x_resnet, x_dino, labels = next(iter(train_loader))

x_resnet = x_resnet.to(device)
x_dino = x_dino.to(device)
labels = labels.to(device)

criterion = nn.CrossEntropyLoss()

# Clear existing gradients
joint_model.zero_grad(set_to_none=True)

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

# Forward pass with mixed precision
with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
    enabled=(device.type == "cuda")
):
    logits = joint_model(x_resnet, x_dino)
    loss = criterion(logits, labels)

# Backpropagate through both networks
loss.backward()

# Verify that gradients reach both backbones
resnet_grad = any(
    p.grad is not None
    for p in joint_model.resnet.parameters()
)

dino_grad = any(
    p.grad is not None
    for p in joint_model.dino.parameters()
)

fusion_grad = any(
    p.grad is not None
    for p in joint_model.classifier.parameters()
)

print("Batch size:", labels.shape[0])
print("Output shape:", logits.shape)
print(f"Loss: {loss.item():.4f}")

print("\nGradient verification:")
print("ResNet18:", resnet_grad)
print("DINOv2:", dino_grad)
print("Fusion classifier:", fusion_grad)

if device.type == "cuda":
    print(
        "\nPeak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

# Clear test gradients
joint_model.zero_grad(set_to_none=True)

Batch size: 32
Output shape: torch.Size([32, 10])
Loss: 2.5833

Gradient verification:
ResNet18: True
DINOv2: True
Fusion classifier: True

Peak GPU memory: 3.57 GB


## 7. Stage 1 — Training the Fusion Classifier

### Objective
Learn how to combine pretrained ResNet18 and DINOv2 representations
before jointly fine-tuning both architectures.

### Architecture
- ResNet18: Frozen (512 features)
- DINOv2 ViT-S/14: Frozen (384 features)
- Fusion classifier: Linear(896, 10), trainable

### Training Configuration
- Optimizer: AdamW
- Fusion learning rate: 1e-4
- Weight decay: 1e-4
- Batch size: 32
- Maximum epochs: 15
- Early stopping patience: 4
- Checkpoint selection: Validation macro-F1

### Data Augmentation
Use the shared training augmentation configured in Step 5.

### Evaluation
- Training: 15,732 images from 18 drivers
- Validation: 3,603 images from 4 unseen drivers
- Metrics: Accuracy, Macro-F1, Cross-Entropy Loss

**Only the fusion classifier is updated during this stage.**

In [9]:
import torch
import torch.nn as nn
from pathlib import Path

# Freeze both pretrained backbones
joint_model.resnet.requires_grad_(False)
joint_model.dino.requires_grad_(False)

# Train only the fusion classifier
joint_model.classifier.requires_grad_(True)

# Optimizer
optimizer = torch.optim.AdamW(
    joint_model.classifier.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

# Experiment directory
fusion_dir = Path(
    "../results/deep_learning/driver_based/"
    "resnet18_dinov2_joint_fusion"
)

fusion_dir.mkdir(parents=True, exist_ok=True)

# Verify
print("ResNet18 trainable:", sum(
    p.numel() for p in joint_model.resnet.parameters()
    if p.requires_grad
))

print("DINOv2 trainable:", sum(
    p.numel() for p in joint_model.dino.parameters()
    if p.requires_grad
))

print("Fusion trainable:", sum(
    p.numel() for p in joint_model.classifier.parameters()
    if p.requires_grad
))

ResNet18 trainable: 0
DINOv2 trainable: 0
Fusion trainable: 8970


### 7.3 Fusion Classifier Training

Train the shared classifier using features extracted from the
frozen ResNet18 and DINOv2 backbones.

After every epoch:
1. Calculate training loss.
2. Evaluate on unseen validation drivers.
3. Calculate accuracy and macro-F1.
4. Save the checkpoint with the highest validation macro-F1.

The pretrained backbones remain unchanged.

In [10]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

torch.manual_seed(42)

max_epochs = 15
patience = 4

best_f1 = -1.0
best_epoch = 0
no_improvement = 0

history = []

checkpoint_path = fusion_dir / "stage1_fusion_head.pt"

use_amp = device.type == "cuda"

total_start = time.time()

for epoch in range(1, max_epochs + 1):

    epoch_start = time.time()

    # Frozen backbones remain in evaluation mode
    joint_model.resnet.eval()
    joint_model.dino.eval()

    # Train fusion classifier
    joint_model.classifier.train()

    train_loss = 0.0

    # --------------------------
    # TRAINING
    # --------------------------
    for x_resnet, x_dino, labels in train_loader:

        x_resnet = x_resnet.to(device)
        x_dino = x_dino.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        # No backbone gradients in Stage 1
        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=use_amp
            ):
                r_features = joint_model.resnet(x_resnet)
                d_features = joint_model.dino(x_dino)

        # Combine features
        combined = torch.cat(
            [r_features.float(), d_features.float()],
            dim=1
        )

        logits = joint_model.classifier(combined)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

    train_loss /= len(train_dataset)

    # --------------------------
    # VALIDATION
    # --------------------------
    joint_model.eval()

    all_preds = []
    all_labels = []

    val_loss = 0.0

    with torch.inference_mode():

        for x_resnet, x_dino, labels in val_loader:

            x_resnet = x_resnet.to(device)
            x_dino = x_dino.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=use_amp
            ):
                logits = joint_model(x_resnet, x_dino)

            loss = criterion(logits.float(), labels)

            val_loss += loss.item() * labels.size(0)

            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    val_loss /= len(val_dataset)

    accuracy = accuracy_score(all_labels, all_preds)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    epoch_time = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": accuracy,
        "val_macro_f1": macro_f1,
        "epoch_seconds": epoch_time
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

    # --------------------------
    # SAVE BEST MODEL
    # --------------------------
    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_epoch = epoch
        no_improvement = 0

        torch.save({
            "classifier_state_dict":
                joint_model.classifier.state_dict(),
            "best_epoch": best_epoch,
            "best_macro_f1": best_f1,
            "best_accuracy": accuracy,
            "seed": 42
        }, checkpoint_path)

        print("Best fusion classifier saved!")

    else:
        no_improvement += 1

    pd.DataFrame(history).to_csv(
        fusion_dir / "stage1_history.csv",
        index=False
    )

    if no_improvement >= patience:
        print("Early stopping triggered.")
        break

print("\nStage 1 completed!")
print("Best epoch:", best_epoch)
print(f"Best validation macro-F1: {best_f1:.4f}")
print(f"Total time: {(time.time()-total_start)/60:.2f} minutes")

# Restore best fusion classifier
best = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

joint_model.classifier.load_state_dict(
    best["classifier_state_dict"]
)

joint_model.eval()

print("Best fusion classifier restored.")

Epoch 01 | Train Loss: 0.1562 | Val Loss: 0.1765 | Accuracy: 0.9553 | Macro-F1: 0.9527 | Time: 253.4s
Best fusion classifier saved!
Epoch 02 | Train Loss: 0.0073 | Val Loss: 0.1584 | Accuracy: 0.9573 | Macro-F1: 0.9552 | Time: 250.5s
Best fusion classifier saved!
Epoch 03 | Train Loss: 0.0041 | Val Loss: 0.1591 | Accuracy: 0.9578 | Macro-F1: 0.9557 | Time: 249.8s
Best fusion classifier saved!
Epoch 04 | Train Loss: 0.0025 | Val Loss: 0.1621 | Accuracy: 0.9603 | Macro-F1: 0.9582 | Time: 249.8s
Best fusion classifier saved!
Epoch 05 | Train Loss: 0.0022 | Val Loss: 0.1633 | Accuracy: 0.9620 | Macro-F1: 0.9600 | Time: 251.6s
Best fusion classifier saved!
Epoch 06 | Train Loss: 0.0018 | Val Loss: 0.1659 | Accuracy: 0.9611 | Macro-F1: 0.9592 | Time: 248.3s
Epoch 07 | Train Loss: 0.0013 | Val Loss: 0.1635 | Accuracy: 0.9611 | Macro-F1: 0.9593 | Time: 250.6s
Epoch 08 | Train Loss: 0.0013 | Val Loss: 0.1648 | Accuracy: 0.9595 | Macro-F1: 0.9575 | Time: 251.7s
Epoch 09 | Train Loss: 0.0008 | Va

In [11]:
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

# Training dataset without augmentation
train_eval_dataset = JointDriverDataset(
    train_df,
    image_root,
    augment=False
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

def evaluate(loader):
    joint_model.eval()

    y_true, y_pred = [], []

    with torch.inference_mode():
        for x_resnet, x_dino, labels in loader:

            x_resnet = x_resnet.to(device)
            x_dino = x_dino.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=(device.type == "cuda")
            ):
                logits = joint_model(x_resnet, x_dino)

            y_true.extend(labels.tolist())
            y_pred.extend(logits.argmax(1).cpu().tolist())

    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro")
    )


train_acc, train_f1 = evaluate(train_eval_loader)
val_acc, val_f1 = evaluate(val_loader)

print(f"Training Accuracy:   {train_acc:.4%}")
print(f"Validation Accuracy: {val_acc:.4%}")
print(f"Training Macro-F1:   {train_f1:.4%}")
print(f"Validation Macro-F1: {val_f1:.4%}")

print("\nGeneralization Gap:")
print(f"Accuracy: {(train_acc-val_acc)*100:.2f} percentage points")
print(f"Macro-F1: {(train_f1-val_f1)*100:.2f} percentage points")

Training Accuracy:   99.9809%
Validation Accuracy: 96.1976%
Training Macro-F1:   99.9802%
Validation Macro-F1: 96.0022%

Generalization Gap:
Accuracy: 3.78 percentage points
Macro-F1: 3.98 percentage points


## 9. Stage 2 — Joint End-to-End Fine-Tuning

### Research Objective
Investigate whether jointly fine-tuning ResNet18 and DINOv2 improves
generalization beyond frozen feature fusion.

### Architecture
- ResNet18: Fully trainable
- DINOv2 ViT-S/14: Fully trainable
- Fusion classifier: Fully trainable
- Loss: Shared Cross-Entropy

### Initialization
Continue from the best Stage 1 fusion model.

### Training Configuration
- Batch size: 32
- Optimizer: AdamW
- ResNet18 learning rate: 3e-6
- DINOv2 learning rate: 1e-6
- Fusion classifier learning rate: 1e-4
- Weight decay: 1e-4
- Maximum epochs: 15
- Early stopping patience: 4
- Mixed precision: BF16

### Regularization
- Shared geometric and photometric augmentation
- Gradient clipping
- Early stopping based on validation macro-F1

### Evaluation
Track training and validation:
1. Accuracy
2. Macro-F1
3. Cross-Entropy Loss

Save the best model separately from Stage 1.

In [12]:
# Unfreeze both pretrained backbones
joint_model.resnet.requires_grad_(True)
joint_model.dino.requires_grad_(True)

# Keep fusion classifier trainable
joint_model.classifier.requires_grad_(True)

# Configure separate learning rates
joint_optimizer = torch.optim.AdamW(
    [
        {
            "params": joint_model.resnet.parameters(),
            "lr": 3e-6
        },
        {
            "params": joint_model.dino.parameters(),
            "lr": 1e-6
        },
        {
            "params": joint_model.classifier.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

# Separate Stage 2 directory
stage2_dir = fusion_dir / "stage2_joint_finetuning"
stage2_dir.mkdir(parents=True, exist_ok=True)

# Verify trainable parameters
for name, module in [
    ("ResNet18", joint_model.resnet),
    ("DINOv2", joint_model.dino),
    ("Fusion Classifier", joint_model.classifier)
]:
    trainable = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    print(f"{name}: {trainable:,} trainable parameters")

print("\nLearning rates:")
for i, group in enumerate(joint_optimizer.param_groups):
    print(f"Group {i + 1}: {group['lr']}")

print("\nJoint fine-tuning configuration ready!")

ResNet18: 11,176,512 trainable parameters
DINOv2: 22,056,576 trainable parameters
Fusion Classifier: 8,970 trainable parameters

Learning rates:
Group 1: 3e-06
Group 2: 1e-06
Group 3: 0.0001

Joint fine-tuning configuration ready!


## 10. Stage 2 — Joint End-to-End Fine-Tuning

### Research Objective
Investigate whether jointly fine-tuning ResNet18 and DINOv2
improves driver-independent classification beyond frozen feature fusion.

### Architecture
- ResNet18: Fully trainable (11.18M parameters)
- DINOv2 ViT-S/14: Fully trainable (22.06M parameters)
- Fusion classifier: Linear(896, 10)
- Total parameters: 33.24 million
- Loss function: Cross-Entropy

### Training Configuration
- Initialization: Best Stage 1 fusion model
- Batch size: 32
- Optimizer: AdamW
- ResNet18 learning rate: 3e-6
- DINOv2 learning rate: 1e-6
- Fusion classifier learning rate: 1e-4
- Weight decay: 1e-4
- Maximum epochs: 15
- Early stopping patience: 4
- Mixed precision: BF16 where supported
- Gradient clipping: 1.0

### Evaluation Metrics
For every epoch, record:
1. Training loss
2. Training accuracy
3. Training macro-F1
4. Validation loss
5. Validation accuracy
6. Validation macro-F1

### Generalization
- Training: 15,732 images from 18 drivers
- Validation: 3,603 images from 4 unseen drivers
- Shared training augmentation
- No validation augmentation

### Model Selection
Save the checkpoint achieving the highest validation macro-F1.

**The Stage 1 checkpoint remains unchanged.**

In [13]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
import time

from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

# ---------------------------------
# 1. Reproducibility
# ---------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Recreate training loader with fixed shuffle seed
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=torch.Generator().manual_seed(SEED)
)

# ---------------------------------
# 2. Restore the best Stage 1 head
# ---------------------------------

stage1_checkpoint = torch.load(
    fusion_dir / "stage1_fusion_head.pt",
    map_location="cpu",
    weights_only=True
)

joint_model.classifier.load_state_dict(
    stage1_checkpoint["classifier_state_dict"]
)

# Enable training for all components
joint_model.requires_grad_(True)

# ---------------------------------
# 3. Training configuration
# ---------------------------------

max_epochs = 15
patience = 4

best_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

history = []

checkpoint_path = stage2_dir / "best_joint_model.pt"

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16
)

criterion = nn.CrossEntropyLoss()

trainable_parameters = [
    p for p in joint_model.parameters()
    if p.requires_grad
]

print("Mixed precision:", amp_dtype if use_amp else "Disabled")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

total_start = time.time()

# ---------------------------------
# 4. Training loop
# ---------------------------------

for epoch in range(1, max_epochs + 1):

    epoch_start = time.time()

    joint_model.train()

    # Keep ResNet18 BatchNorm running statistics fixed.
    # Its affine parameters remain trainable.
    for module in joint_model.resnet.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()

    running_loss = 0.0

    train_predictions = []
    train_labels = []

    # ========== TRAINING ==========

    for batch_idx, (x_resnet, x_dino, labels) in enumerate(
        train_loader, 1
    ):

        x_resnet = x_resnet.to(device, non_blocking=True)
        x_dino = x_dino.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        joint_optimizer.zero_grad(set_to_none=True)

        # Forward pass
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp
        ):

            logits = joint_model(x_resnet, x_dino)

            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss detected.")

        # Backpropagation through BOTH backbones
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(joint_optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=1.0
        )

        # Update parameters
        scaler.step(joint_optimizer)
        scaler.update()

        # Record training results
        running_loss += loss.item() * labels.size(0)

        predictions = logits.argmax(dim=1)

        train_predictions.extend(
            predictions.detach().cpu().tolist()
        )

        train_labels.extend(
            labels.cpu().tolist()
        )

        if batch_idx % 100 == 0:
            print(
                f"Epoch {epoch:02d} | "
                f"Batch {batch_idx}/{len(train_loader)}"
            )

    # ========== TRAINING METRICS ==========

    train_loss = running_loss / len(train_dataset)

    train_acc = accuracy_score(
        train_labels,
        train_predictions
    )

    train_f1 = f1_score(
        train_labels,
        train_predictions,
        average="macro",
        zero_division=0
    )

    # ========== VALIDATION ==========

    joint_model.eval()

    val_predictions = []
    val_labels = []

    val_loss_total = 0.0

    with torch.inference_mode():

        for x_resnet, x_dino, labels in val_loader:

            x_resnet = x_resnet.to(device, non_blocking=True)
            x_dino = x_dino.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=amp_dtype,
                enabled=use_amp
            ):

                logits = joint_model(
                    x_resnet,
                    x_dino
                )

                loss = criterion(logits, labels)

            val_loss_total += (
                loss.item() * labels.size(0)
            )

            predictions = logits.argmax(dim=1)

            val_predictions.extend(
                predictions.cpu().tolist()
            )

            val_labels.extend(
                labels.cpu().tolist()
            )

    # ========== VALIDATION METRICS ==========

    val_loss = val_loss_total / len(val_dataset)

    val_acc = accuracy_score(
        val_labels,
        val_predictions
    )

    val_f1 = f1_score(
        val_labels,
        val_predictions,
        average="macro",
        zero_division=0
    )

    epoch_time = time.time() - epoch_start

    # ========== RECORD RESULTS ==========

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "epoch_seconds": epoch_time
    })

    print(f"\n{'=' * 65}")
    print(f"EPOCH {epoch:02d}")
    print(f"{'=' * 65}")

    print(
        f"Training   | Loss: {train_loss:.4f} | "
        f"Accuracy: {train_acc:.4%} | "
        f"Macro-F1: {train_f1:.4%}"
    )

    print(
        f"Validation | Loss: {val_loss:.4f} | "
        f"Accuracy: {val_acc:.4%} | "
        f"Macro-F1: {val_f1:.4%}"
    )

    print(f"Epoch time: {epoch_time:.1f}s")

    # ========== SAVE BEST MODEL ==========

    if val_f1 > best_f1:

        best_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": joint_model.state_dict(),
            "best_epoch": best_epoch,
            "best_val_accuracy": val_acc,
            "best_val_macro_f1": best_f1,
            "seed": SEED,
            "stage": "joint_end_to_end"
        }, checkpoint_path)

        print("New best joint model saved!")

    else:
        epochs_without_improvement += 1

    # Save history after every epoch
    pd.DataFrame(history).to_csv(
        stage2_dir / "training_history.csv",
        index=False
    )

    # ========== EARLY STOPPING ==========

    if epochs_without_improvement >= patience:
        print("\nEarly stopping triggered.")
        break

# ---------------------------------
# 5. Final summary
# ---------------------------------

total_time = time.time() - total_start

print("\n" + "=" * 65)
print("JOINT FINE-TUNING COMPLETED")
print("=" * 65)

print("Best epoch:", best_epoch)
print(f"Best validation macro-F1: {best_f1:.4%}")
print(f"Total training time: {total_time / 60:.2f} minutes")

# Restore best checkpoint
best_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

joint_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

joint_model.eval()

print("Best joint model restored.")

Mixed precision: torch.bfloat16
Training batches: 492
Validation batches: 113
Epoch 01 | Batch 100/492
Epoch 01 | Batch 200/492
Epoch 01 | Batch 300/492
Epoch 01 | Batch 400/492

EPOCH 01
Training   | Loss: 0.0012 | Accuracy: 99.9682% | Macro-F1: 99.9679%
Validation | Loss: 0.2282 | Accuracy: 95.7258% | Macro-F1: 95.4784%
Epoch time: 274.4s
New best joint model saved!
Epoch 02 | Batch 100/492
Epoch 02 | Batch 200/492
Epoch 02 | Batch 300/492
Epoch 02 | Batch 400/492

EPOCH 02
Training   | Loss: 0.0004 | Accuracy: 99.9809% | Macro-F1: 99.9797%
Validation | Loss: 0.2722 | Accuracy: 95.5315% | Macro-F1: 95.2816%
Epoch time: 274.1s
Epoch 03 | Batch 100/492
Epoch 03 | Batch 200/492
Epoch 03 | Batch 300/492
Epoch 03 | Batch 400/492

EPOCH 03
Training   | Loss: 0.0008 | Accuracy: 99.9873% | Macro-F1: 99.9871%
Validation | Loss: 0.2743 | Accuracy: 95.6148% | Macro-F1: 95.4278%
Epoch time: 275.6s
Epoch 04 | Batch 100/492
Epoch 04 | Batch 200/492


KeyboardInterrupt: 